In [19]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc

In [20]:
"""
Self-contained module: all auxiliary functions (wavelengths r and b),
the v2-isolated splits, and the contraction maps H_1, H_2, H_3.

Only dependency: numpy. No local module imports -- everything is in this file,
using the same function bodies translated previously.

Input convention (same as before):
    array = [t, u, v]
    t   = array[0]          (scalar)
    u_1 = array[1][0]
    u_2 = array[1][1]
    v_1 = array[2][0]
    v_2 = array[2][1]

Vector-valued maps return numpy arrays of shape (2,).
The H maps take two such arrays: I_r = [t_r, u_r, v_r], I_b = [t_b, u_b, v_b].
"""
# ---------------------------------------------------------------------------
# Parameters -- set these to your problem's values
# ---------------------------------------------------------------------------
n_r = 1.5143      # refractive index for wavelength r, must satisfy n_r > 1
n_b = 1.5224      # refractive index for wavelength b, must satisfy n_b > 1
rho_0 = 2*10**(-5)    # rho_0 > 0
d_0 = 1.0      # d_0 > 0
k_0 = rho_0 / d_0
w = np.array([0.0, 1.0])   # fixed vector w = (0, 1)


def _unpack(array):
    """Split the input array into t, u_1, u_2, v_1, v_2."""
    t = array[0]
    u_1 = array[1][0]
    u_2 = array[1][1]
    v_1 = array[2][0]
    v_2 = array[2][1]
    return t, u_1, u_2, v_1, v_2


# ===========================================================================
# ===========================================================================
#                       WAVELENGTH r FUNCTIONS
# ===========================================================================
# ===========================================================================

def alpha(array):
    """alpha(u) = (u_1 - sqrt((u_1^2+u_2^2)(n_r^2-1) + u_1^2)) / (u_1^2+u_2^2)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    norm_sq = u_1**2 + u_2**2
    root = np.sqrt(norm_sq * (n_r**2 - 1) + u_1**2)
    return (u_1 - root) / norm_sq


def M_r(array):
    """
    M_r(t,u) = (1/n_r) [ (sin t, cos t)
                         - alpha(u) (u_1 sin t - u_2 cos t,
                                     u_1 cos t + u_2 sin t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha(array)
    vec1 = np.array([np.sin(t), np.cos(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_1 * np.cos(t) + u_2 * np.sin(t)])
    return (vec1 - a * vec2) / n_r


def D_r(array):
    """D_r(t,u) = ((n_r-1) d_0 - u_1 (1 - cos t)) / (n_r - w . M_r(t,u))."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mr = M_r(array)
    numerator = (n_r - 1) * d_0 - u_1 * (1 - np.cos(t))
    denominator = n_r - np.dot(w, Mr)
    return numerator / denominator


def F_r(array):
    """F_r(t,u) = u_1 (sin t, cos t) + D_r(t,u) M_r(t,u)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    return u_1 * np.array([np.sin(t), np.cos(t)]) + D_r(array) * M_r(array)


def Lambda_2r(array):
    """Lambda_{2,r}(t,u) = sqrt(1 + 1/n_r^2 - (2/n_r) w . M_r(t,u))."""
    Mr = M_r(array)
    return np.sqrt(1.0 + 1.0 / n_r**2 - (2.0 / n_r) * np.dot(w, Mr))


def alpha_tilde(array):
    """
    alpha_tilde(u,v) = (n_r^2-1) *
        ( v_1 + ((u_1 v_1 + u_2 v_2)(n_r^2-1) + u_1 v_1) / R ) * (1/(u_1 + R))^2,
    with R = sqrt((u_1^2+u_2^2)(n_r^2-1) + u_1^2).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    root = np.sqrt((u_1**2 + u_2**2) * (n_r**2 - 1) + u_1**2)
    middle = v_1 + ((u_1 * v_1 + u_2 * v_2) * (n_r**2 - 1) + u_1 * v_1) / root
    return (n_r**2 - 1) * middle * (1.0 / (u_1 + root))**2


def M_r_tilde(array):
    """
    M_r_tilde(t,u,v) = (1/n_r) [ (cos t, -sin t)
        - alpha_tilde(u,v) (u_1 sin t - u_2 cos t,  u_2 sin t + u_1 cos t)
        - alpha(u) ((u_1 - v_2) cos t + (u_2 + v_1) sin t,
                    (v_2 - u_1) sin t + (v_1 + u_2) cos t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha(array)
    a_t = alpha_tilde(array)
    vec1 = np.array([np.cos(t), -np.sin(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3 = np.array([(u_1 - v_2) * np.cos(t) + (u_2 + v_1) * np.sin(t),
                     (v_2 - u_1) * np.sin(t) + (v_1 + u_2) * np.cos(t)])
    return (vec1 - a_t * vec2 - a * vec3) / n_r


def D_r_tilde(array):
    """
    D_r_tilde(t,u,v) =
        [ (v_1 (cos t - 1) - u_1 sin t)(n_r - w . M_r)
          + ((n_r-1) d_0 - u_1 (1 - cos t)) (w . M_r_tilde) ]
        / (n_r - w . M_r)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    w_dot_Mr = np.dot(w, M_r(array))
    w_dot_Mr_t = np.dot(w, M_r_tilde(array))
    numerator = ((v_1 * (np.cos(t) - 1) - u_1 * np.sin(t)) * (n_r - w_dot_Mr)
                 + ((n_r - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Mr_t)
    return numerator / (n_r - w_dot_Mr)**2


def F_r_tilde(array):
    """
    F_r_tilde(t,u,v) = u_1 (cos t, -sin t) + v_1 (sin t, cos t)
                       + M_r_tilde(t,u,v) D_r(t,u) + M_r(t,u) D_r_tilde(t,u,v).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    term1 = u_1 * np.array([np.cos(t), -np.sin(t)])
    term2 = v_1 * np.array([np.sin(t), np.cos(t)])
    return term1 + term2 + M_r_tilde(array) * D_r(array) + M_r(array) * D_r_tilde(array)


def Lambda_2r_tilde(array):
    """Lambda_2r_tilde(t,u,v) = -(1/n_r) (w . M_r_tilde) / Lambda_2r(t,u)."""
    return -(1.0 / n_r) * np.dot(w, M_r_tilde(array)) / Lambda_2r(array)


# ===========================================================================
# ===========================================================================
#                       WAVELENGTH b FUNCTIONS
# ===========================================================================
# ===========================================================================

def alpha_b(array):
    """alpha_b(u) = (u_1 - sqrt((u_1^2+u_2^2)(n_b^2-1) + u_1^2)) / (u_1^2+u_2^2)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    norm_sq = u_1**2 + u_2**2
    root = np.sqrt(norm_sq * (n_b**2 - 1) + u_1**2)
    return (u_1 - root) / norm_sq


def M_b(array):
    """
    M_b(t,u) = (1/n_b) [ (sin t, cos t)
                         - alpha_b(u) (u_1 sin t - u_2 cos t,
                                       u_1 cos t + u_2 sin t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_b(array)
    vec1 = np.array([np.sin(t), np.cos(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_1 * np.cos(t) + u_2 * np.sin(t)])
    return (vec1 - a * vec2) / n_b


def D_b(array):
    """D_b(t,u) = ((n_b-1) d_0 - u_1 (1 - cos t)) / (n_b - w . M_b(t,u))."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    Mb = M_b(array)
    numerator = (n_b - 1) * d_0 - u_1 * (1 - np.cos(t))
    denominator = n_b - np.dot(w, Mb)
    return numerator / denominator


def F_b(array):
    """F_b(t,u) = u_1 (sin t, cos t) + D_b(t,u) M_b(t,u)."""
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    return u_1 * np.array([np.sin(t), np.cos(t)]) + D_b(array) * M_b(array)


def Lambda_2b(array):
    """Lambda_{2,b}(t,u) = sqrt(1 + 1/n_b^2 - (2/n_b) w . M_b(t,u))."""
    Mb = M_b(array)
    return np.sqrt(1.0 + 1.0 / n_b**2 - (2.0 / n_b) * np.dot(w, Mb))


def alpha_tilde_b(array):
    """
    alpha_tilde_b(u,v) = (n_b^2-1) *
        ( v_1 + ((u_1 v_1 + u_2 v_2)(n_b^2-1) + u_1 v_1) / R ) * (1/(u_1 + R))^2,
    with R = sqrt((u_1^2+u_2^2)(n_b^2-1) + u_1^2).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    root = np.sqrt((u_1**2 + u_2**2) * (n_b**2 - 1) + u_1**2)
    middle = v_1 + ((u_1 * v_1 + u_2 * v_2) * (n_b**2 - 1) + u_1 * v_1) / root
    return (n_b**2 - 1) * middle * (1.0 / (u_1 + root))**2


def M_b_tilde(array):
    """
    M_b_tilde(t,u,v) = (1/n_b) [ (cos t, -sin t)
        - alpha_tilde_b(u,v) (u_1 sin t - u_2 cos t,  u_2 sin t + u_1 cos t)
        - alpha_b(u) ((u_1 - v_2) cos t + (u_2 + v_1) sin t,
                      (v_2 - u_1) sin t + (v_1 + u_2) cos t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha_b(array)
    a_t = alpha_tilde_b(array)
    vec1 = np.array([np.cos(t), -np.sin(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3 = np.array([(u_1 - v_2) * np.cos(t) + (u_2 + v_1) * np.sin(t),
                     (v_2 - u_1) * np.sin(t) + (v_1 + u_2) * np.cos(t)])
    return (vec1 - a_t * vec2 - a * vec3) / n_b


def D_b_tilde(array):
    """
    D_b_tilde(t,u,v) =
        [ (v_1 (cos t - 1) - u_1 sin t)(n_b - w . M_b)
          + ((n_b-1) d_0 - u_1 (1 - cos t)) (w . M_b_tilde) ]
        / (n_b - w . M_b)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    w_dot_Mb = np.dot(w, M_b(array))
    w_dot_Mb_t = np.dot(w, M_b_tilde(array))
    numerator = ((v_1 * (np.cos(t) - 1) - u_1 * np.sin(t)) * (n_b - w_dot_Mb)
                 + ((n_b - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Mb_t)
    return numerator / (n_b - w_dot_Mb)**2


def F_b_tilde(array):
    """
    F_b_tilde(t,u,v) = u_1 (cos t, -sin t) + v_1 (sin t, cos t)
                       + M_b_tilde(t,u,v) D_b(t,u) + M_b(t,u) D_b_tilde(t,u,v).
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    term1 = u_1 * np.array([np.cos(t), -np.sin(t)])
    term2 = v_1 * np.array([np.sin(t), np.cos(t)])
    return term1 + term2 + M_b_tilde(array) * D_b(array) + M_b(array) * D_b_tilde(array)


def Lambda_2b_tilde(array):
    """Lambda_2b_tilde(t,u,v) = -(1/n_b) (w . M_b_tilde) / Lambda_2b(t,u)."""
    return -(1.0 / n_b) * np.dot(w, M_b_tilde(array)) / Lambda_2b(array)


# ===========================================================================
# ===========================================================================
#         v2-ISOLATED SPLITS (wavelength r):
#         f_tilde(t,u,v) = v_2 * f_tilde_v2 + f_tilde_others
# ===========================================================================
# ===========================================================================

def alpha_tilde_v2(array):
    """
    Coefficient of v_2 in alpha_tilde:
    alpha_tilde_v2(u) = (n_r^2-1) * (u_2 (n_r^2-1) / R) * (1/(u_1+R))^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    R = np.sqrt((u_1**2 + u_2**2) * (n_r**2 - 1) + u_1**2)
    return (n_r**2 - 1) * (u_2 * (n_r**2 - 1) / R) * (1.0 / (u_1 + R))**2


def alpha_tilde_others(array):
    """
    v_2-free part of alpha_tilde (= alpha_tilde with v_2 set to 0):
    (n_r^2-1) * ( v_1 + (u_1 v_1 (n_r^2-1) + u_1 v_1)/R ) * (1/(u_1+R))^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    R = np.sqrt((u_1**2 + u_2**2) * (n_r**2 - 1) + u_1**2)
    middle = v_1 + (u_1 * v_1 * (n_r**2 - 1) + u_1 * v_1) / R
    return (n_r**2 - 1) * middle * (1.0 / (u_1 + R))**2


def M_r_tilde_v2(array):
    """
    M_r_tilde_v2(t,u,v) = (1/n_r) [ - alpha_tilde_v2(u) (u_1 sin t - u_2 cos t,
                                                         u_2 sin t + u_1 cos t)
                                    - alpha(u) (-cos t, sin t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha(array)
    a_v2 = alpha_tilde_v2(array)
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3_v2 = np.array([-np.cos(t), np.sin(t)])
    return (-a_v2 * vec2 - a * vec3_v2) / n_r


def M_r_tilde_others(array):
    """
    M_r_tilde_others(t,u,v) = (1/n_r) [ (cos t, -sin t)
        - alpha_tilde_others(u,v) (u_1 sin t - u_2 cos t,  u_2 sin t + u_1 cos t)
        - alpha(u) (u_1 cos t + (u_2+v_1) sin t,  -u_1 sin t + (v_1+u_2) cos t) ].
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    a = alpha(array)
    a_oth = alpha_tilde_others(array)
    vec1 = np.array([np.cos(t), -np.sin(t)])
    vec2 = np.array([u_1 * np.sin(t) - u_2 * np.cos(t),
                     u_2 * np.sin(t) + u_1 * np.cos(t)])
    vec3_oth = np.array([u_1 * np.cos(t) + (u_2 + v_1) * np.sin(t),
                         -u_1 * np.sin(t) + (v_1 + u_2) * np.cos(t)])
    return (vec1 - a_oth * vec2 - a * vec3_oth) / n_r


def D_r_tilde_v2(array):
    """
    D_r_tilde_v2 = ((n_r-1) d_0 - u_1 (1 - cos t)) (w . M_r_tilde_v2)
                   / (n_r - w . M_r)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    w_dot_Mr = np.dot(w, M_r(array))
    w_dot_Mv2 = np.dot(w, M_r_tilde_v2(array))
    return ((n_r - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Mv2 / (n_r - w_dot_Mr)**2


def D_r_tilde_others(array):
    """
    D_r_tilde_others = [ (v_1 (cos t - 1) - u_1 sin t)(n_r - w . M_r)
                         + ((n_r-1) d_0 - u_1 (1 - cos t)) (w . M_r_tilde_others) ]
                       / (n_r - w . M_r)^2.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    w_dot_Mr = np.dot(w, M_r(array))
    w_dot_Moth = np.dot(w, M_r_tilde_others(array))
    numerator = ((v_1 * (np.cos(t) - 1) - u_1 * np.sin(t)) * (n_r - w_dot_Mr)
                 + ((n_r - 1) * d_0 - u_1 * (1 - np.cos(t))) * w_dot_Moth)
    return numerator / (n_r - w_dot_Mr)**2


def F_r_tilde_v2(array):
    """F_r_tilde_v2 = M_r_tilde_v2 * D_r + M_r * D_r_tilde_v2."""
    return M_r_tilde_v2(array) * D_r(array) + M_r(array) * D_r_tilde_v2(array)


def F_r_tilde_others(array):
    """
    F_r_tilde_others = u_1 (cos t, -sin t) + v_1 (sin t, cos t)
                       + M_r_tilde_others * D_r + M_r * D_r_tilde_others.
    """
    t, u_1, u_2, v_1, v_2 = _unpack(array)
    term1 = u_1 * np.array([np.cos(t), -np.sin(t)])
    term2 = v_1 * np.array([np.sin(t), np.cos(t)])
    return (term1 + term2
            + M_r_tilde_others(array) * D_r(array)
            + M_r(array) * D_r_tilde_others(array))


def Lambda_2r_tilde_v2(array):
    """Lambda_2r_tilde_v2 = -(1/n_r) (w . M_r_tilde_v2) / Lambda_2r."""
    return -(1.0 / n_r) * np.dot(w, M_r_tilde_v2(array)) / Lambda_2r(array)


def Lambda_2r_tilde_others(array):
    """Lambda_2r_tilde_others = -(1/n_r) (w . M_r_tilde_others) / Lambda_2r."""
    return -(1.0 / n_r) * np.dot(w, M_r_tilde_others(array)) / Lambda_2r(array)


# ===========================================================================
# ===========================================================================
#               CONTRACTION MAPS H_1, H_2, H_3
#               I_r = [t_r, u_r, v_r],  I_b = [t_b, u_b, v_b]
# ===========================================================================
# ===========================================================================

def H_1(I_r, I_b):
    """H_1(I_r, I_b) = F_r_tilde_1(I_r) / F_b_tilde_1(I_b)."""
    return F_r_tilde(I_r)[0] / F_b_tilde(I_b)[0]


def H_2(I_r, I_b):
    """H_2(I_r, I_b) = u_{r2}  (second component of u_r)."""
    return I_r[1][1]


def H_3(I_r, I_b):
    """
    H_3(I_r, I_b) = the value of v_2 (= z_3'(t)) isolated from

        F_r_tilde_1(I_r) (M_r1(I_r) Lambda_2b_tilde(I_b) - M_b1_tilde(I_b) Lambda_2r(I_r))
      = F_b_tilde_1(I_b) (M_r1_tilde(I_r) Lambda_2b(I_b) - M_b1(I_b) Lambda_2r_tilde(I_r)),

    namely

        [ F_b_tilde_1(I_b) (M_r1_tilde_others(I_r) Lambda_2b(I_b)
                            - M_b1(I_b) Lambda_2r_tilde_others(I_r))
          - F_r_tilde_1_others(I_r) (M_r1(I_r) Lambda_2b_tilde(I_b)
                                     - M_b1_tilde(I_b) Lambda_2r(I_r)) ]
        /
        [ F_r_tilde_1_v2(I_r) (M_r1(I_r) Lambda_2b_tilde(I_b)
                               - M_b1_tilde(I_b) Lambda_2r(I_r))
          - F_b_tilde_1(I_b) (M_r1_tilde_v2(I_r) Lambda_2b(I_b)
                              - M_b1(I_b) Lambda_2r_tilde_v2(I_r)) ].
    """
    # --- pieces evaluated at I_r (wavelength r) ---
    Mr1 = M_r(I_r)[0]                          # M_r1(I_r)
    Lam_r = Lambda_2r(I_r)                     # Lambda_2r(I_r)
    Mr1_oth = M_r_tilde_others(I_r)[0]         # M_r1_tilde_others(I_r)
    Mr1_v2 = M_r_tilde_v2(I_r)[0]              # M_r1_tilde_v2(I_r)
    LamT_r_oth = Lambda_2r_tilde_others(I_r)   # Lambda_2r_tilde_others(I_r)
    LamT_r_v2 = Lambda_2r_tilde_v2(I_r)        # Lambda_2r_tilde_v2(I_r)
    Fr1_oth = F_r_tilde_others(I_r)[0]         # F_r1_tilde_others(I_r)
    Fr1_v2 = F_r_tilde_v2(I_r)[0]              # F_r1_tilde_v2(I_r)

    # --- pieces evaluated at I_b (wavelength b, full functions) ---
    Fb1_t = F_b_tilde(I_b)[0]                  # F_b1_tilde(I_b)
    Mb1 = M_b(I_b)[0]                          # M_b1(I_b)
    Mb1_t = M_b_tilde(I_b)[0]                  # M_b1_tilde(I_b)
    Lam_b = Lambda_2b(I_b)                     # Lambda_2b(I_b)
    LamT_b = Lambda_2b_tilde(I_b)              # Lambda_2b_tilde(I_b)

    # common bracket appearing in numerator and denominator
    bracket = Mr1 * LamT_b - Mb1_t * Lam_r     # M_r1 Lambda_2b_tilde - M_b1_tilde Lambda_2r

    numerator = (Fb1_t * (Mr1_oth * Lam_b - Mb1 * LamT_r_oth)
                 - Fr1_oth * bracket)
    denominator = (Fr1_v2 * bracket
                   - Fb1_t * (Mr1_v2 * Lam_b - Mb1 * LamT_r_v2))
    return numerator / denominator


# ===========================================================================
# Sanity checks
# ===========================================================================
if __name__ == "__main__":
    rng = np.random.default_rng(0)

    print("=== Linearity check: f_tilde == v2 * f_v2 + f_others (random points) ===")
    for trial in range(5):
        t = rng.uniform(-1, 1)
        u = rng.uniform(0.5, 2.0, size=2)
        v = rng.uniform(-1, 1, size=2)
        arr = [t, u, v]
        v2 = v[1]

        checks = {
            "alpha_tilde": (alpha_tilde(arr),
                            v2 * alpha_tilde_v2(arr) + alpha_tilde_others(arr)),
            "M_r_tilde":   (M_r_tilde(arr),
                            v2 * M_r_tilde_v2(arr) + M_r_tilde_others(arr)),
            "D_r_tilde":   (D_r_tilde(arr),
                            v2 * D_r_tilde_v2(arr) + D_r_tilde_others(arr)),
            "F_r_tilde":   (F_r_tilde(arr),
                            v2 * F_r_tilde_v2(arr) + F_r_tilde_others(arr)),
            "Lambda_2r_tilde": (Lambda_2r_tilde(arr),
                                v2 * Lambda_2r_tilde_v2(arr) + Lambda_2r_tilde_others(arr)),
        }
        ok = all(np.allclose(full, split, rtol=1e-12, atol=1e-12)
                 for full, split in checks.values())
        print(f"  trial {trial}: all splits match -> {ok}")

    print()
    print("=== H_3 consistency check ===")
    print("Plug v_r2 = H_3(I_r, I_b) back into I_r; then the identity")
    print("  F_r_tilde_1 (M_r1 LamT_b - M_b1_t Lam_r) "
          "= F_b_tilde_1 (M_r1_tilde Lam_b - M_b1 LamT_r)")
    print("should hold exactly.")
    for trial in range(3):
        I_r = [rng.uniform(-1, 1), rng.uniform(0.5, 2.0, 2), rng.uniform(-1, 1, 2)]
        I_b = [rng.uniform(-1, 1), rng.uniform(0.5, 2.0, 2), rng.uniform(-1, 1, 2)]

        v2_star = H_3(I_r, I_b)
        I_r_star = [I_r[0], I_r[1], np.array([I_r[2][0], v2_star])]

        lhs = (F_r_tilde(I_r_star)[0]
               * (M_r(I_r_star)[0] * Lambda_2b_tilde(I_b)
                  - M_b_tilde(I_b)[0] * Lambda_2r(I_r_star)))
        rhs = (F_b_tilde(I_b)[0]
               * (M_r_tilde(I_r_star)[0] * Lambda_2b(I_b)
                  - M_b(I_b)[0] * Lambda_2r_tilde(I_r_star)))
        print(f"  trial {trial}: v2* = {v2_star: .6f}, residual = {lhs - rhs: .3e}")

    print()
    print("=== H maps smoke test ===")
    I_r = [0.3, np.array([1.0, 0.5]), np.array([0.2, -0.1])]
    I_b = [0.25, np.array([1.1, 0.4]), np.array([0.15, -0.05])]
    print("H_1 =", H_1(I_r, I_b))
    print("H_2 =", H_2(I_r, I_b))
    print("H_3 =", H_3(I_r, I_b))

=== Linearity check: f_tilde == v2 * f_v2 + f_others (random points) ===
  trial 0: all splits match -> True
  trial 1: all splits match -> True
  trial 2: all splits match -> True
  trial 3: all splits match -> True
  trial 4: all splits match -> True

=== H_3 consistency check ===
Plug v_r2 = H_3(I_r, I_b) back into I_r; then the identity
  F_r_tilde_1 (M_r1 LamT_b - M_b1_t Lam_r) = F_b_tilde_1 (M_r1_tilde Lam_b - M_b1 LamT_r)
should hold exactly.
  trial 0: v2* =  17.754094, residual = -2.220e-16
  trial 1: v2* =  13.899936, residual = -1.388e-16
  trial 2: v2* =  3.103788, residual = -5.551e-17

=== H maps smoke test ===
H_1 = 0.9469701933402497
H_2 = 0.5
H_3 = 4.487908844364674


In [21]:
# ===========================================================================
# Admissible point P  (eq:admissible P)
# ===========================================================================

def Delta_r():
    """Delta_r = n_r / (n_r - 1)."""
    return n_r / (n_r - 1)


def Delta_b():
    """Delta_b = n_b / (n_b - 1)."""
    return n_b / (n_b - 1)


def P(k_0):
    """
    P = ( (Db - Dr + S) / (Dr - Db + S),
          0,
          ((Db + Dr + S) / 2) * rho_0 ),
    where Dr = Delta_r, Db = Delta_b, k_0 = rho_0 / d_0,
    and S = sqrt((Db - Dr)^2 - 4 k_0 Db Dr).
    Returns np.array([P_1, P_2, P_3]).
    """
    Dr = Delta_r()
    Db = Delta_b()
    # k_0 = rho_0 / d_0

    discriminant = (Db - Dr)**2 - 4.0 * k_0 * Db * Dr
    if discriminant < 0:
        raise ValueError(
            f"Negative discriminant ({discriminant:.6e}): "
            "(Delta_b - Delta_r)^2 - 4 k_0 Delta_b Delta_r < 0 for the "
            "current n_r, n_b, rho_0, d_0 -- P is not real."
        )
    S = np.sqrt(discriminant)

    P_1 = (Db - Dr + S) / (Dr - Db + S)
    P_2 = 0.0
    P_3 = ((Db + Dr + S) / 2.0) * rho_0

    return np.array([P_1, P_2, P_3])

In [22]:
def H(I_r,I_b):
    # t= I_r[0]
    # z_2 = I_r[1]
    # z_3 = I_r[2]

    # z_1 = I_b[0]
    # z_2_z1 = I_b[1]
    # z_3_z1 = I_b[2]

    return np.array([H_1(I_r,I_b), H_2(I_r,I_b), H_3(I_r,I_b)])

In [ ]:
def findingsolution(k_0, n, tol):

    array = np.ones(3)*t*P(k_0)